# 4. Final Model — Two-Stage XGBoost (Aug 2024 leave-month-out validation)


## Validation strategy

I'm predicting **August 2025**, so the most honest validation month is **August 2024** — the calendar analog of the target. I tried Jul 2025 validation earlier and got bitten by calibration transfer: Jul's residual bias pattern didn't match Aug's, so the isotonic calibrators I fit on Jul made August *worse* on Portfolio C and D.

Aug 2024 has the same calendar structure as Aug 2025 (no federal holidays, summer season, same weekday distribution). Any per-portfolio bias I find there is far more likely to generalize to Aug 2025 than a July bias is.

## What I'm shipping

**Stage 1 (daily), pooled per metric:**
- **CV:** `reg:quantileerror`, quantile α ∈ [0.52, 0.65] learned by Optuna
- **CCT:** `reg:squarederror`
- **Abandon Rate → Abandoned Calls:** `count:poisson` on the count target (because Ridge on the raw rate hit alpha=1389 in NB2, and the count has a distribution Poisson can handle)

**Per-portfolio isotonic calibration on Aug 2024 residuals.** Fits `sklearn.IsotonicRegression` per (metric, portfolio) on the holdout-model's Aug 2024 predictions vs actuals. Applied to the final model's Aug 2025 predictions at submission time. This is the piece that matters — it's the whole reason for switching to Aug 2024 validation.

**Seasonal-naive blend for Abandoned Calls only.** `Aug 2024 × (Jan–Jul 2025 / Jan–Jul 2024) ratio`, blended with the (calibrated) XGB prediction using weights from NB5 §4: A=0.75, B=0.65, C=0.95, D=0.85. CV and CCT are left alone.

**Stage 2 (interval):** Same DOW × half-hour profile machinery as before. Abandon rate profile is rescaled per day to match Stage-1's derived daily rate.

## What I tried and rejected earlier

- **Per-portfolio XGBoost.** Regressed on all 4 portfolios. Pooled training with `portfolio` as a categorical feature was already giving per-portfolio branching where needed, and per-portfolio models had 4× less training data per model.
- **Monotone constraints.** Slight regression on CV and ABD in the last run. Removed — the tree was already learning the right relationships without the constraint and the constraint was over-restricting edge-case splits.
- **Erlang-A queueing for ABD.** Staffing is so high relative to volume (1–3% utilization) that queue dynamics don't drive abandonment in this data. Wrong physical model for the problem.
- **Holiday-adjacency feature.** NB3 §4's evidence was n=4 from one single holiday.
- **Jul 2025 validation with isotonic calibration.** The reason for this whole refactor.

## Success criteria

1. CV and CCT Aug 2024 holdout WMAPE comparable to prior Jul 2025 holdout numbers (4–7% CV, 3–4% CCT).
2. Isotonic calibrators move Aug 2024 bias toward 0 (by construction) AND the calibrated Aug 2025 predictions land closer to historical than the pooled-without-calibration numbers I had before.
3. Derived August 2025 abandon rate within ±25% of historical per portfolio (A ~0.011, B ~0.017, C ~0.012, D ~0.013).
4. Submission CSV: 0 NaN, 0 negatives, 0 abandoned-calls/rate mismatches.

Output: `submission_forecast.csv`.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.metrics import mean_absolute_error
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_FILE = "Data for Datathon (Revised).xlsx"
TEMPLATE_FILE = "template_forecast_v00.csv"
PORTFOLIOS = ['A', 'B', 'C', 'D']
MONTH_MAP = {'January': 1, 'February': 2, 'March': 3, 'April': 4,
             'May': 5, 'June': 6, 'July': 7, 'August': 8,
             'September': 9, 'October': 10, 'November': 11, 'December': 12}

US_HOLIDAYS = set([
    datetime(2024, 1, 1), datetime(2024, 1, 15), datetime(2024, 2, 19),
    datetime(2024, 5, 27), datetime(2024, 7, 4), datetime(2024, 9, 2),
    datetime(2024, 10, 14), datetime(2024, 11, 11), datetime(2024, 11, 28),
    datetime(2024, 12, 25),
    datetime(2025, 1, 1), datetime(2025, 1, 20), datetime(2025, 2, 17),
    datetime(2025, 5, 26), datetime(2025, 7, 4), datetime(2025, 9, 1),
    datetime(2025, 10, 13), datetime(2025, 11, 11), datetime(2025, 11, 27),
    datetime(2025, 12, 25)
])

## 1. Load Data

In [ ]:
daily_data = {}
for p in PORTFOLIOS:
    df = pd.read_excel(DATA_FILE, sheet_name=f"{p} - Daily")
    df['Date'] = pd.to_datetime(df['Date'].str.strip().str[:8], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df['dow'] = df['Date'].dt.dayofweek
    df['month'] = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
    df['is_weekend'] = (df['dow'] >= 5).astype(int)
    df['is_holiday'] = df['Date'].isin(US_HOLIDAYS).astype(int)
    df['year'] = df['Date'].dt.year
    df['day_of_year'] = df['Date'].dt.dayofyear
    df['is_month_start'] = (df['day_of_month'] <= 3).astype(int)
    df['is_month_end'] = (df['day_of_month'] >= 28).astype(int)
    df['is_friday'] = (df['dow'] == 4).astype(int)
    df['is_monday'] = (df['dow'] == 0).astype(int)
    # Day after holiday
    df['is_day_after_holiday'] = df['Date'].apply(
        lambda d: int((d - timedelta(days=1)) in US_HOLIDAYS))
    daily_data[p] = df

interval_data = {}
for p in PORTFOLIOS:
    df = pd.read_excel(DATA_FILE, sheet_name=f"{p} - Interval")
    df = df.dropna(subset=['Interval', 'Call Volume']).reset_index(drop=True)
    df['Interval'] = df['Interval'].astype(str)
    def parse_interval(val):
        val = str(val).strip()
        if 'days' in val:
            return val.split(' ')[-1][:5]
        return val[:5] if len(val) >= 5 else val
    df['Interval_str'] = df['Interval'].apply(parse_interval)
    df['half_hour'] = df['Interval_str'].apply(
        lambda x: int(x.split(':')[0]) * 2 + (1 if int(x.split(':')[1]) >= 30 else 0))
    df['month_num'] = df['Month'].map(MONTH_MAP)
    dates = []
    for _, row in df.iterrows():
        try:
            dates.append(datetime(2025, row['month_num'], row['Day']))
        except ValueError:
            dates.append(None)
    df['Date'] = pd.to_datetime(dates, format='mixed')
    df = df.dropna(subset=['Date']).reset_index(drop=True)
    df['dow'] = df['Date'].dt.dayofweek
    interval_data[p] = df

staffing = pd.read_excel(DATA_FILE, sheet_name="Daily Staffing")
staffing.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
staffing['Date'] = pd.to_datetime(staffing['Date'], format='mixed')

print("Loaded.")
for p in PORTFOLIOS:
    print(f"  {p}: {len(daily_data[p])} daily ({daily_data[p]['Date'].min().date()} to {daily_data[p]['Date'].max().date()}), {len(interval_data[p])} interval")

## 2. Stage 1: Daily Feature Engineering

In [ ]:
# Add Abandoned Calls as a count target (Poisson-friendly), since modeling the raw
# Abandon Rate with squared loss collapses to "predict the mean" (Ridge baseline hit alpha=1389).
for p in PORTFOLIOS:
    daily_data[p]['Abandoned Calls'] = (
        daily_data[p]['Call Volume'] * daily_data[p]['Abandon Rate']
    )

daily_metrics = ['Call Volume', 'CCT', 'Abandon Rate', 'Abandoned Calls']

def build_daily_features(df, staffing_df, portfolio_label, portfolio_id):
    """Build features for daily-level model using 2 years of data."""
    d = df.copy()
    d['portfolio'] = portfolio_id

    # Lag features
    for metric in daily_metrics:
        for lag in [1, 7, 14, 28, 365]:
            d[f'{metric}_lag{lag}'] = d[metric].shift(lag)
        # Rolling features
        for window in [7, 14, 28]:
            d[f'{metric}_roll{window}'] = d[metric].rolling(window).mean().shift(1)
            d[f'{metric}_rollstd{window}'] = d[metric].rolling(window).std().shift(1)
        # Same DOW last week / 2 weeks ago
        d[f'{metric}_same_dow_1w'] = d[metric].shift(7)
        d[f'{metric}_same_dow_2w'] = d[metric].shift(14)
        d[f'{metric}_same_dow_avg'] = (d[metric].shift(7) + d[metric].shift(14)) / 2

    # Merge staffing
    staff = staffing_df[['Date', portfolio_label]].copy()
    staff.columns = ['Date', 'staffing']
    d = d.merge(staff, on='Date', how='left')

    return d

all_daily = []
for i, p in enumerate(PORTFOLIOS):
    feat = build_daily_features(daily_data[p], staffing, p, i)
    all_daily.append(feat)

daily_df = pd.concat(all_daily, ignore_index=True)
print(f"Daily training data: {daily_df.shape}")
print(f"Date range: {daily_df['Date'].min().date()} to {daily_df['Date'].max().date()}")


## 3. Stage 1: Define Features & Train/Val Split

In [ ]:
base_features = ['dow', 'month', 'day_of_month', 'week_of_year',
                  'is_weekend', 'is_holiday', 'year', 'day_of_year',
                  'is_month_start', 'is_month_end', 'is_friday', 'is_monday',
                  'is_day_after_holiday', 'portfolio']

cv_features = base_features + [
    'Call Volume_lag1', 'Call Volume_lag7', 'Call Volume_lag14', 'Call Volume_lag28', 'Call Volume_lag365',
    'Call Volume_roll7', 'Call Volume_roll14', 'Call Volume_roll28',
    'Call Volume_rollstd7',
    'Call Volume_same_dow_avg',
    'staffing'
]

cct_features = base_features + [
    'CCT_lag7', 'CCT_lag14', 'CCT_lag28', 'CCT_lag365',
    'CCT_roll7', 'CCT_roll14', 'CCT_roll28',
    'CCT_same_dow_avg',
    'staffing'
]

# ABD is modeled as a COUNT (Abandoned Calls), not a rate.
abd_features = base_features + [
    'Abandoned Calls_lag7', 'Abandoned Calls_lag14', 'Abandoned Calls_lag28', 'Abandoned Calls_lag365',
    'Abandoned Calls_roll7', 'Abandoned Calls_roll14', 'Abandoned Calls_roll28',
    'Abandoned Calls_same_dow_avg',
    'Call Volume_lag7', 'Call Volume_lag14',  # volume pressure drives abandonment
    'Call Volume_roll7',
    'staffing'
]

DAILY_TARGETS = {'cv': 'Call Volume', 'cct': 'CCT', 'abd': 'Abandoned Calls'}
DAILY_FEATURES = {'cv': cv_features, 'cct': cct_features, 'abd': abd_features}

# Validation window: Aug 2024 (calendar analog of the Aug 2025 target).
# Leave-month-out: training is everything EXCEPT Aug 2024. Final retrain in cell 12
# uses the full dataset including Aug 2024 for predicting Aug 2025.
AUG_2024_START = datetime(2024, 8, 1)
AUG_2024_END   = datetime(2024, 9, 1)

val_mask   = (daily_df['Date'] >= AUG_2024_START) & (daily_df['Date'] < AUG_2024_END)
train_mask = ~val_mask

print(f"Train: {train_mask.sum()} rows (everything except Aug 2024)")
print(f"Val:   {val_mask.sum()} rows (Aug 2024)")
print(f"Val date range: {daily_df[val_mask]['Date'].min().date()} to {daily_df[val_mask]['Date'].max().date()}")


## 4. Stage 1: Optuna Tuning + Train Daily Models

In [ ]:
# CV folds for Optuna tuning — leave-month-out on three months within 2024.
# Fold 3 (Aug 2024) is the primary target; May and Jun 2024 are secondary signal.
# In every fold, Aug 2024 is excluded from training so the tuner never "sees" the
# validation-analog month.

def leave_month_out_fold(val_start, val_end):
    val = (daily_df['Date'] >= val_start) & (daily_df['Date'] < val_end)
    # Always exclude Aug 2024 from training (even when it's not the val month)
    excl_aug = (daily_df['Date'] >= AUG_2024_START) & (daily_df['Date'] < AUG_2024_END)
    train = (~val) & (~excl_aug)
    return {'train': train, 'val': val}

daily_cv_folds = [
    leave_month_out_fold(datetime(2024, 5, 1), datetime(2024, 6, 1)),  # Fold 1: May 2024
    leave_month_out_fold(datetime(2024, 6, 1), datetime(2024, 7, 1)),  # Fold 2: Jun 2024
    leave_month_out_fold(datetime(2024, 8, 1), datetime(2024, 9, 1)),  # Fold 3: Aug 2024 (primary)
]

for i, fold in enumerate(daily_cv_folds):
    tr_dates = daily_df[fold['train']]['Date']
    va_dates = daily_df[fold['val']]['Date']
    print(f"Fold {i+1}: Train {fold['train'].sum()} rows ({tr_dates.min().date()}-{tr_dates.max().date()}), "
          f"Val {fold['val'].sum()} rows ({va_dates.min().date()}-{va_dates.max().date()})")


In [ ]:
def daily_cv_objective(trial, metric_key, target_col, features, folds, df):
    params = {
        'verbosity': 0, 'nthread': -1, 'seed': 42, 'enable_categorical': True,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 60),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 0, 5.0),
    }
    if metric_key == 'cv':
        alpha = trial.suggest_float('quantile_alpha', 0.52, 0.65)
        params['objective'] = 'reg:quantileerror'
        params['quantile_alpha'] = alpha
        params['eval_metric'] = 'mae'
    elif metric_key == 'abd':
        params['objective'] = 'count:poisson'
        params['eval_metric'] = 'mae'
    else:
        params['objective'] = 'reg:squarederror'
        params['eval_metric'] = 'mae'

    fold_scores = []
    for fold in folds:
        X_tr = df[fold['train']][features].copy()
        y_tr = df[fold['train']][target_col].copy()
        X_va = df[fold['val']][features].copy()
        y_va = df[fold['val']][target_col].copy()
        valid_tr, valid_va = y_tr.notna(), y_va.notna()
        X_tr, y_tr = X_tr[valid_tr], y_tr[valid_tr]
        X_va, y_va = X_va[valid_va], y_va[valid_va]
        if len(X_va) == 0 or len(X_tr) == 0:
            continue
        X_tr['portfolio'] = X_tr['portfolio'].astype('category')
        X_va['portfolio'] = X_va['portfolio'].astype('category')
        # Explicit base_score to avoid XGBoost's auto-init underflowing to -inf
        # on Poisson (log-link) folds with pathological target distributions.
        y_mean = float(y_tr.mean())
        params['base_score'] = max(y_mean, 1e-3) if np.isfinite(y_mean) and y_mean > 0 else 1.0
        dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
        dval = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
        model = xgb.train(params, dtrain, num_boost_round=1000,
                          evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
        pred = np.clip(model.predict(dval), 0, None)
        fold_scores.append(np.mean(np.abs(y_va.values - pred)))
    return np.mean(fold_scores) if fold_scores else 1e9


N_TRIALS = 80
MAX_ROUNDS = {'cv': 1000, 'cct': 1000, 'abd': 1500}

daily_best_params = {}
daily_models = {}

for metric_key, target_col in DAILY_TARGETS.items():
    features = DAILY_FEATURES[metric_key]
    print(f'\n===== {metric_key.upper()} (pooled, Aug 2024 leave-month-out) =====')

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(
        lambda trial: daily_cv_objective(trial, metric_key, target_col, features, daily_cv_folds, daily_df),
        n_trials=N_TRIALS)

    bp = study.best_params.copy()
    if metric_key == 'cv':
        alpha = bp.pop('quantile_alpha')
        bp['objective'] = 'reg:quantileerror'
        bp['quantile_alpha'] = alpha
        print(f'  Quantile alpha: {alpha:.4f}')
    elif metric_key == 'abd':
        bp['objective'] = 'count:poisson'
    else:
        bp['objective'] = 'reg:squarederror'
    bp.update({'eval_metric': 'mae', 'verbosity': 0, 'nthread': -1, 'seed': 42,
               'enable_categorical': True})
    daily_best_params[metric_key] = bp

    # Aug 2024 holdout fit (train_mask = everything except Aug 2024, val_mask = Aug 2024)
    X_tr = daily_df[train_mask][features].copy()
    y_tr = daily_df[train_mask][target_col].copy()
    X_va = daily_df[val_mask][features].copy()
    y_va = daily_df[val_mask][target_col].copy()
    valid_tr, valid_va = y_tr.notna(), y_va.notna()
    X_tr, y_tr = X_tr[valid_tr], y_tr[valid_tr]
    X_va, y_va = X_va[valid_va], y_va[valid_va]
    X_tr['portfolio'] = X_tr['portfolio'].astype('category')
    X_va['portfolio'] = X_va['portfolio'].astype('category')
    y_mean = float(y_tr.mean())
    bp['base_score'] = max(y_mean, 1e-3) if np.isfinite(y_mean) and y_mean > 0 else 1.0

    dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
    dval = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
    model = xgb.train(bp, dtrain, num_boost_round=MAX_ROUNDS[metric_key],
                      evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)

    val_pred = np.clip(model.predict(dval), 0, None)
    mae = np.mean(np.abs(y_va.values - val_pred))
    denom = np.sum(np.abs(y_va.values))
    wmape = (np.sum(np.abs(y_va.values - val_pred)) / denom) if denom > 0 else float('nan')
    bias = np.mean(val_pred - y_va.values)
    daily_models[metric_key] = model

    print(f'  CV best MAE (Optuna avg across 3 folds): {study.best_value:.2f}')
    print(f'  Aug 2024 holdout: iter={model.best_iteration}, MAE={mae:.2f}, WMAPE={wmape:.4f}, Bias={bias:+.2f}')
    for pid, plabel in enumerate(PORTFOLIOS):
        pmask = X_va['portfolio'] == pid
        if pmask.sum() > 0:
            p_mae = np.mean(np.abs(y_va.values[pmask] - val_pred[pmask]))
            p_den = np.sum(np.abs(y_va.values[pmask]))
            p_wmape = (np.sum(np.abs(y_va.values[pmask] - val_pred[pmask])) / p_den) if p_den > 0 else float('nan')
            p_bias = np.mean(val_pred[pmask] - y_va.values[pmask])
            print(f'    {plabel}: MAE={p_mae:.2f}, WMAPE={p_wmape:.4f}, Bias={p_bias:+.2f}')


# ==== Per-portfolio isotonic calibration on Aug 2024 holdout ====
# The whole point of validating on Aug 2024 rather than Jul 2025: the calibrators
# now learn Aug-specific bias patterns that should transfer to Aug 2025.
from sklearn.isotonic import IsotonicRegression

calibrators = {metric_key: {} for metric_key in DAILY_TARGETS}

print('\n===== Isotonic calibration (Aug 2024 holdout, per portfolio) =====')
for metric_key, target_col in DAILY_TARGETS.items():
    features = DAILY_FEATURES[metric_key]
    model = daily_models[metric_key]

    X_va_all = daily_df[val_mask][features].copy()
    y_va_all = daily_df[val_mask][target_col].copy()
    pf_raw   = daily_df[val_mask]['portfolio'].values
    valid = y_va_all.notna()
    X_va_all = X_va_all[valid]
    y_va_all = y_va_all[valid]
    pf_raw   = pf_raw[valid.values]
    X_va_all['portfolio'] = X_va_all['portfolio'].astype('category')
    dval = xgb.DMatrix(X_va_all, label=y_va_all, enable_categorical=True)
    val_pred = np.clip(model.predict(dval), 0, None)

    print(f'\n{metric_key.upper()}:')
    for pid, plabel in enumerate(PORTFOLIOS):
        m = pf_raw == pid
        if m.sum() < 10:
            print(f'  {plabel}: too few points ({m.sum()}), skipping (calibrator = identity)')
            calibrators[metric_key][pid] = None
            continue
        p_pred = val_pred[m]
        p_actual = y_va_all.values[m]
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0)
        iso.fit(p_pred, p_actual)
        calibrators[metric_key][pid] = iso
        p_cal = iso.predict(p_pred)
        raw_mae = float(np.mean(np.abs(p_actual - p_pred)))
        cal_mae = float(np.mean(np.abs(p_actual - p_cal)))
        raw_bias = float(np.mean(p_pred - p_actual))
        cal_bias = float(np.mean(p_cal - p_actual))
        print(f'  {plabel}: MAE {raw_mae:.2f} → {cal_mae:.2f} '
              f'| bias {raw_bias:+.2f} → {cal_bias:+.2f}')


## 5. Stage 1: Retrain on All Data & Predict August Daily

In [ ]:
# Retrain final models on the FULL daily dataset (including Aug 2024).
# The calibrators were fit on a different model (trained on everything EXCEPT Aug 2024)
# and are now applied to this full-data model's Aug 2025 predictions. The two models
# are similar but not identical — this is an approximation that works because they share
# 95%+ of their training data.
final_daily_models = {}

for metric_key, target_col in DAILY_TARGETS.items():
    features = DAILY_FEATURES[metric_key]
    X = daily_df[features].copy()
    y = daily_df[target_col].copy()
    v = y.notna()
    X, y = X[v], y[v]
    X['portfolio'] = X['portfolio'].astype('category')

    bp = dict(daily_best_params[metric_key])
    y_mean_full = float(y.mean())
    bp['base_score'] = max(y_mean_full, 1e-3) if np.isfinite(y_mean_full) and y_mean_full > 0 else 1.0

    best_iter = max(daily_models[metric_key].best_iteration, 10)
    dfull = xgb.DMatrix(X, label=y, enable_categorical=True)
    model = xgb.train(bp, dfull, num_boost_round=best_iter)
    final_daily_models[metric_key] = model
    print(f"{metric_key.upper()} (pooled, full data): Retrained on {len(X)} rows, {best_iter} rounds")

# Predict August 2025 daily values, applying per-portfolio isotonic calibration
august_daily_preds = {p: {} for p in PORTFOLIOS}
for pid, p in enumerate(PORTFOLIOS):
    d = daily_data[p].copy()
    aug_rows = []
    for day in range(1, 32):
        aug_date = datetime(2025, 8, day)
        row = {
            'Date': aug_date,
            'dow': aug_date.weekday(),
            'month': 8,
            'day_of_month': day,
            'week_of_year': aug_date.isocalendar()[1],
            'is_weekend': int(aug_date.weekday() >= 5),
            'is_holiday': int(aug_date in US_HOLIDAYS),
            'year': 2025,
            'day_of_year': aug_date.timetuple().tm_yday,
            'is_month_start': int(day <= 3),
            'is_month_end': int(day >= 28),
            'is_friday': int(aug_date.weekday() == 4),
            'is_monday': int(aug_date.weekday() == 0),
            'is_day_after_holiday': int((aug_date - timedelta(days=1)) in US_HOLIDAYS),
            'portfolio': pid,
        }
        for metric in daily_metrics:
            for lag in [1, 7, 14, 28, 365]:
                lag_date = aug_date - timedelta(days=lag)
                lag_row = d[d['Date'] == lag_date]
                row[f'{metric}_lag{lag}'] = lag_row[metric].values[0] if len(lag_row) > 0 else np.nan
            for window in [7, 14, 28]:
                end = aug_date - timedelta(days=1)
                start = end - timedelta(days=window-1)
                w = d[(d['Date'] >= start) & (d['Date'] <= end)][metric]
                row[f'{metric}_roll{window}'] = w.mean() if len(w) > 0 else np.nan
                row[f'{metric}_rollstd{window}'] = w.std() if len(w) > 1 else np.nan
            row[f'{metric}_same_dow_1w'] = row[f'{metric}_lag7']
            row[f'{metric}_same_dow_2w'] = row[f'{metric}_lag14']
            l7 = row.get(f'{metric}_lag7', np.nan)
            l14 = row.get(f'{metric}_lag14', np.nan)
            vals = [v for v in [l7, l14] if not (isinstance(v, float) and np.isnan(v))]
            row[f'{metric}_same_dow_avg'] = np.mean(vals) if vals else np.nan
        staff_row = staffing[staffing['Date'] == aug_date]
        if len(staff_row) > 0:
            row['staffing'] = staff_row[p].values[0]
        else:
            same_dow = staffing[staffing['Date'].dt.weekday == aug_date.weekday()].sort_values('Date')
            row['staffing'] = same_dow[p].iloc[-1] if len(same_dow) > 0 else np.nan
        aug_rows.append(row)

    aug_daily = pd.DataFrame(aug_rows)

    for metric_key, target_col in DAILY_TARGETS.items():
        features = DAILY_FEATURES[metric_key]
        X_aug = aug_daily[features].copy()
        X_aug['portfolio'] = X_aug['portfolio'].astype('category')
        daug = xgb.DMatrix(X_aug, enable_categorical=True)
        raw_pred = np.clip(final_daily_models[metric_key].predict(daug), 0, None)

        # Apply per-portfolio isotonic calibration (fit on Aug 2024 residuals)
        cal = calibrators[metric_key].get(pid)
        if cal is not None:
            pred = np.clip(cal.predict(raw_pred), 0, None)
        else:
            pred = raw_pred
        august_daily_preds[p][target_col] = pred

    # Derive Abandon Rate from calibrated counts
    cv_pred = august_daily_preds[p]['Call Volume']
    abd_calls_pred = august_daily_preds[p]['Abandoned Calls']
    august_daily_preds[p]['Abandon Rate'] = np.clip(
        abd_calls_pred / np.maximum(cv_pred, 1.0), 0.0, 1.0)

    august_daily_preds[p]['Date'] = aug_daily['Date']
    august_daily_preds[p]['dow'] = aug_daily['dow']

    print(f"\nPortfolio {p} August 2025 daily (calibrated):")
    print(f"  CV: mean={august_daily_preds[p]['Call Volume'].mean():.0f}")
    print(f"  CCT: mean={august_daily_preds[p]['CCT'].mean():.1f}")
    print(f"  ABD Calls: mean={august_daily_preds[p]['Abandoned Calls'].mean():.1f}")
    print(f"  ABD Rate (derived): mean={august_daily_preds[p]['Abandon Rate'].mean():.4f}")


## 5b. Seasonal-naive blend for Abandoned Calls (only)

Notebook 5 §4 grid-searched the `w` in `blend = w × XGB + (1−w) × seasonal_naive` on the Jul 2025 holdout per portfolio. For Abandoned Calls the picked weights were:

| Portfolio | Picked w | Count WMAPE improvement (Jul) |
|---|---|---|
| A | 0.75 | 6% |
| B | 0.65 | **16%** |
| C | 0.95 | <1% |
| D | 0.85 | 4% |

The seasonal-naive predictor is `aug_2024 × (Jan–Jul 2025 mean / Jan–Jul 2024 mean)` — one scalar YoY ratio per portfolio, applied to last August's daily values.

**CV and CCT are NOT blended.** CV was already near ceiling (4.3–7.2% WMAPE) with a marginal 2% blend improvement — not worth the risk. CCT is already at 3.1–4.0% WMAPE. The blend is applied only where it actually matters: Portfolio A and B Abandoned Calls, which are the weak spots in the Stage-1 ABD pipeline.

After blending, `Abandon Rate` is re-derived from the blended count divided by Stage-1 Call Volume prediction, so the daily rate flowing into Stage-2 disaggregation (cell 16) is consistent with the new blended count.


In [ ]:
# Seasonal-naive projection for Abandoned Calls per portfolio.
# Returns a 31-element numpy array indexed by day-of-month 1..31, with NaN
# for missing or NaN days — the blend will fall back to XGB for those days.
def seasonal_naive_aug_abd_calls(portfolio_label):
    d = daily_data[portfolio_label]
    h1_24 = d[(d['year']==2024) & (d['month'].isin(range(1,8)))]['Abandoned Calls'].mean()
    h1_25 = d[(d['year']==2025) & (d['month'].isin(range(1,8)))]['Abandoned Calls'].mean()
    ratio = h1_25 / h1_24 if (pd.notna(h1_24) and h1_24 > 0) else 1.0
    out = np.full(31, np.nan)
    aug24 = d[(d['year']==2024) & (d['month']==8)]
    for _, row in aug24.iterrows():
        dom = int(row['day_of_month'])
        v = row['Abandoned Calls']
        if 1 <= dom <= 31 and pd.notna(v):
            out[dom - 1] = v * ratio
    return out, ratio

# Blend weights from 5_experiments.ipynb §4 (Jul-holdout grid search).
# Keyed by portfolio label.
ABD_BLEND_W = {'A': 0.75, 'B': 0.65, 'C': 0.95, 'D': 0.85}

print('=== Seasonal-naive blend for Abandoned Calls (Stage-1 patch) ===')
header = f"{'port':<6} {'xgb_mean':>9} {'naive_mean':>11} {'ratio':>7} {'w_xgb':>7} {'blend':>9} {'rate_before':>12} {'rate_after':>12} {'hist_rate':>11} {'n_fb':>5}"
print(header)
print('-' * len(header))

for pid, p in enumerate(PORTFOLIOS):
    xgb_abd = np.asarray(august_daily_preds[p]['Abandoned Calls'], dtype=float)
    naive_abd, ratio = seasonal_naive_aug_abd_calls(p)

    # Both arrays are length 31 (day-of-month aligned). Any NaN in naive_abd means
    # Aug 2024 was missing or had NaN on that calendar day — fall back to XGB.
    assert len(xgb_abd) == 31, f'Expected 31 Aug days for XGB pred on {p}, got {len(xgb_abd)}'
    assert len(naive_abd) == 31, f'Expected 31 Aug days for seasonal-naive on {p}, got {len(naive_abd)}'

    w = ABD_BLEND_W[p]
    nan_mask = np.isnan(naive_abd)
    n_fallback = int(nan_mask.sum())

    # Blend where naive is valid, XGB alone where it's not
    blended = np.where(
        nan_mask,
        xgb_abd,
        w * xgb_abd + (1 - w) * np.where(nan_mask, 0, naive_abd),  # inner where avoids NaN×0 warnings
    )
    blended = np.clip(blended, 0.0, None)

    cv_pred = np.asarray(august_daily_preds[p]['Call Volume'], dtype=float)
    old_rate = float(np.mean(np.clip(xgb_abd / np.maximum(cv_pred, 1.0), 0.0, 1.0)))
    new_rate = float(np.mean(np.clip(blended / np.maximum(cv_pred, 1.0), 0.0, 1.0)))
    hist_rate = float(daily_data[p]['Abandon Rate'].mean())

    # Use nanmean for naive print in case of fallback days
    naive_mean_str = float(np.nanmean(naive_abd)) if not np.all(nan_mask) else float('nan')
    print(f'{p:<6} {xgb_abd.mean():>9.1f} {naive_mean_str:>11.1f} {ratio:>7.3f} {w:>7.2f} '
          f'{blended.mean():>9.1f} {old_rate:>12.4f} {new_rate:>12.4f} {hist_rate:>11.4f} {n_fallback:>5d}')

    # In-place update: cell 16 reads from august_daily_preds[p] for both Abandoned Calls and Abandon Rate.
    august_daily_preds[p]['Abandoned Calls'] = blended
    august_daily_preds[p]['Abandon Rate'] = np.clip(blended / np.maximum(cv_pred, 1.0), 0.0, 1.0)

    # Safety: fail fast if any NaN slipped through
    assert not np.any(np.isnan(august_daily_preds[p]['Abandoned Calls'])), f'NaN in blended Abandoned Calls for {p}'
    assert not np.any(np.isnan(august_daily_preds[p]['Abandon Rate'])), f'NaN in derived Abandon Rate for {p}'

print('\nStage-1 Abandoned Calls replaced with blended predictions.')
print('Cell 16 will now rescale the Stage-2 abandon rate profile against the blended daily rates.')
print('n_fb = number of August days where Aug 2024 was missing/NaN and blend fell back to XGB alone.')


## 6. Stage 2: Learn Interval Disaggregation Profiles

In [ ]:
# Build disaggregation profiles from COMPLETE days only (48 intervals)
# Then normalize so proportions sum to exactly 1.0 per DOW

cv_profiles = {}
cct_profiles = {}
abd_profiles = {}

for p in PORTFOLIOS:
    idf = interval_data[p].copy()
    
    # Only keep complete days (exactly 48 intervals)
    day_counts = idf.groupby('Date').size()
    complete_days = day_counts[day_counts == 48].index
    idf_complete = idf[idf['Date'].isin(complete_days)].copy()
    print(f'{p}: {len(complete_days)}/{len(day_counts)} complete days used for profiles')
    
    # If too few complete days, use all days but fill missing slots with 0
    if len(complete_days) < 14:
        print(f'  WARNING: Too few complete days, using all days with gap-filling')
        # Create full grid
        all_dates = idf['Date'].unique()
        full_grid = pd.MultiIndex.from_product([all_dates, range(48)], names=['Date', 'half_hour'])
        idf_complete = idf.set_index(['Date', 'half_hour']).reindex(full_grid).reset_index()
        idf_complete['Call Volume'] = idf_complete['Call Volume'].fillna(0)
        idf_complete['CCT'] = idf_complete['CCT'].fillna(idf['CCT'].mean())
        idf_complete['Abandoned Rate'] = idf_complete['Abandoned Rate'].fillna(0)
        idf_complete['dow'] = idf_complete['Date'].dt.dayofweek
    
    # Call Volume proportions
    daily_totals = idf_complete.groupby('Date')['Call Volume'].sum().reset_index()
    daily_totals.columns = ['Date', 'daily_total']
    idf_complete = idf_complete.merge(daily_totals, on='Date')
    idf_complete['cv_proportion'] = np.where(
        idf_complete['daily_total'] > 0,
        idf_complete['Call Volume'] / idf_complete['daily_total'],
        1.0 / 48  # uniform if no calls
    )
    
    # Average proportion by DOW x half_hour
    cv_raw = idf_complete.groupby(['dow', 'half_hour'])['cv_proportion'].mean()
    
    # NORMALIZE: ensure each DOW sums to exactly 1.0
    cv_normalized = {}
    for dow in range(7):
        dow_props = {hh: cv_raw.get((dow, hh), 1/48) for hh in range(48)}
        total = sum(dow_props.values())
        for hh in range(48):
            cv_normalized[(dow, hh)] = dow_props[hh] / total if total > 0 else 1/48
    cv_profiles[p] = cv_normalized
    
    # CCT: average by DOW x half_hour (not proportional)
    cct_vals = idf_complete[idf_complete['CCT'].notna()].groupby(['dow', 'half_hour'])['CCT'].mean().to_dict()
    cct_profiles[p] = cct_vals
    
    # Abandon Rate: average by DOW x half_hour
    abd_vals = idf_complete[idf_complete['Abandoned Rate'].notna()].groupby(['dow', 'half_hour'])['Abandoned Rate'].mean().to_dict()
    abd_profiles[p] = abd_vals
    
    # Verify
    for dow in range(7):
        total = sum(cv_normalized.get((dow, hh), 0) for hh in range(48))
        if abs(total - 1.0) > 0.001:
            print(f'  ERROR: {p} DOW={dow} sums to {total:.6f}')

print('\nAll profiles normalized to sum=1.0 per DOW.')

## 7. Combine: Daily Predictions x Interval Profiles

In [ ]:
template = pd.read_csv(TEMPLATE_FILE)

def interval_to_halfhour(s):
    h, m = int(s.split(':')[0]), int(s.split(':')[1])
    return h * 2 + (1 if m >= 30 else 0)

template['half_hour'] = template['Interval'].apply(interval_to_halfhour)

# Precompute each portfolio/DOW's historical profile-implied daily rate
# (volume-weighted average of the DOW x half_hour abandon rate profile).
# We use this to rescale the profile so the daily total matches Stage-1.
hist_profile_daily_rate = {}
for p in PORTFOLIOS:
    hist_profile_daily_rate[p] = {}
    for dow in range(7):
        r = 0.0
        for hh in range(48):
            r += abd_profiles[p].get((dow, hh), 0.0) * cv_profiles[p].get((dow, hh), 0.0)
        hist_profile_daily_rate[p][dow] = r

for p in PORTFOLIOS:
    preds = august_daily_preds[p]

    for _, row in template.iterrows():
        day = row['Day']
        hh = row['half_hour']
        idx = row.name

        aug_date = datetime(2025, 8, day)
        dow = aug_date.weekday()
        day_idx = day - 1  # 0-indexed

        # Call Volume: daily total x proportion for this DOW x half_hour
        daily_cv = preds['Call Volume'][day_idx]
        proportion = cv_profiles[p].get((dow, hh), 0)
        interval_cv = max(0, round(daily_cv * proportion))

        # CCT: use DOW x half_hour average directly
        interval_cct = cct_profiles[p].get((dow, hh), preds['CCT'][day_idx])
        interval_cct = max(0, round(interval_cct, 2))

        # Abandon Rate: keep the historical DOW x hh *shape* but rescale so the
        # volume-weighted daily rate matches Stage-1's derived daily rate.
        # This puts Stage-1 (Poisson on counts) in charge of the level, and
        # the historical profile in charge of the intra-day shape.
        stage1_daily_rate = float(preds['Abandon Rate'][day_idx])
        hist_daily = hist_profile_daily_rate[p][dow]
        if hist_daily > 1e-9:
            scale = stage1_daily_rate / hist_daily
        else:
            scale = 0.0
        hist_interval_rate = abd_profiles[p].get((dow, hh), stage1_daily_rate)
        interval_abd = hist_interval_rate * scale
        interval_abd = max(0.0, min(1.0, interval_abd))

        # Abandoned Calls = Rate x Volume
        interval_abd_calls = max(0, round(interval_abd * interval_cv))

        template.loc[idx, f'Calls_Offered_{p}'] = int(interval_cv)
        template.loc[idx, f'CCT_{p}'] = round(interval_cct, 2)
        template.loc[idx, f'Abandoned_Rate_{p}'] = round(interval_abd, 6)
        template.loc[idx, f'Abandoned_Calls_{p}'] = int(interval_abd_calls)

template = template.drop(columns=['half_hour'])

# Cast types
for p in PORTFOLIOS:
    template[f'Calls_Offered_{p}'] = template[f'Calls_Offered_{p}'].astype(int)
    template[f'Abandoned_Calls_{p}'] = template[f'Abandoned_Calls_{p}'].astype(int)

# Validate
print(f"Shape: {template.shape}")
print(f"NaN: {template.isna().sum().sum()}")
numeric = template.select_dtypes(include='number')
print(f"Negatives: {(numeric < 0).sum().sum()}")

template.to_csv('submission_forecast.csv', index=False)
print("Saved to submission_forecast.csv")


## 8. Sanity Checks

In [ ]:
sub = pd.read_csv('submission_forecast.csv')

print("DAILY CALL VOLUME TOTALS (Aug predictions vs historical):")
print(f"{'Portfolio':<10} {'Aug Daily Avg':>15} {'Hist Daily Avg':>15} {'Aug 2024 Avg':>15} {'Ratio (hist)':>12}")
print("-" * 70)
for p in PORTFOLIOS:
    aug_daily = sub.groupby('Day')[f'Calls_Offered_{p}'].sum().mean()
    hist_daily = daily_data[p]['Call Volume'].mean()
    # August 2024 specifically
    aug24 = daily_data[p][(daily_data[p]['month'] == 8) & (daily_data[p]['year'] == 2024)]['Call Volume'].mean()
    print(f"{p:<10} {aug_daily:>15.0f} {hist_daily:>15.0f} {aug24:>15.0f} {aug_daily/hist_daily:>12.2f}")

print("\nWEEKDAY vs WEEKEND:")
for p in PORTFOLIOS:
    wd_days = [d for d in range(1,32) if datetime(2025,8,d).weekday() < 5]
    we_days = [d for d in range(1,32) if datetime(2025,8,d).weekday() >= 5]
    wd_avg = sub[sub['Day'].isin(wd_days)].groupby('Day')[f'Calls_Offered_{p}'].sum().mean()
    we_avg = sub[sub['Day'].isin(we_days)].groupby('Day')[f'Calls_Offered_{p}'].sum().mean()
    print(f"  {p}: Weekday={wd_avg:.0f}, Weekend={we_avg:.0f}, ratio={we_avg/wd_avg:.2f}")

print("\nCCT AVERAGES:")
for p in PORTFOLIOS:
    aug_cct = sub[f'CCT_{p}'].mean()
    hist_cct = daily_data[p]['CCT'].mean()
    print(f"  {p}: Aug={aug_cct:.1f}s, Historical={hist_cct:.1f}s")

print("\nABANDON RATE (daily weighted avg):")
for p in PORTFOLIOS:
    cv = sub[f'Calls_Offered_{p}']
    abd = sub[f'Abandoned_Rate_{p}']
    weighted = np.average(abd, weights=cv) if cv.sum() > 0 else 0
    hist_abd = daily_data[p]['Abandon Rate'].mean()
    print(f"  {p}: Aug={weighted:.4f}, Historical={hist_abd:.4f}")

print("\nABANDONED CALLS CONSISTENCY:")
for p in PORTFOLIOS:
    cv = sub[f'Calls_Offered_{p}']
    abd_r = sub[f'Abandoned_Rate_{p}']
    abd_c = sub[f'Abandoned_Calls_{p}']
    expected = np.round(abd_r * cv).astype(int)
    mismatch = (abd_c != expected).sum()
    print(f"  {p}: {mismatch} mismatches")

In [ ]:
# Visualize
fig, axes = plt.subplots(4, 3, figsize=(20, 16))
pred_cols = [(f'Calls_Offered', 'Call Volume'), (f'CCT', 'CCT'), (f'Abandoned_Rate', 'Abandon Rate')]

for j, p in enumerate(PORTFOLIOS):
    for i, (col_prefix, label) in enumerate(pred_cols):
        ax = axes[j, i]
        col = f'{col_prefix}_{p}'
        for day in range(1, 32):
            day_data = sub[sub['Day'] == day]
            dow = datetime(2025, 8, day).weekday()
            color = 'blue' if dow < 5 else 'red'
            ax.plot(range(48), day_data[col].values, color=color, alpha=0.3, linewidth=0.5)
        ax.set_xlabel('Half-hour slot')
        if i == 0:
            ax.set_ylabel(f'Portfolio {p}')
        if j == 0:
            ax.set_title(label)
        ax.grid(True, alpha=0.3)

fig.suptitle('August 2025 Two-Stage Predictions (blue=weekday, red=weekend)', fontsize=14)
plt.tight_layout()
plt.savefig('eda_plots/10_twostage_august_predictions.png', dpi=150)
plt.show()
print("Done.")

## 9. Conclusions

### Path

Ridge baseline → pooled XGBoost (Jul 2025 val) → per-portfolio XGB (tried, regressed, reverted) → pooled + monotone + isotonic on Jul 2025 val (tried, isotonic backfired on Aug because Jul bias didn't transfer) → **pooled XGBoost + Aug 2024 leave-month-out validation + isotonic calibration fit on Aug 2024 + seasonal-naive blend for Abandoned Calls**.

### The key change this round

Switched validation from Jul 2025 to Aug 2024. The previous run showed that isotonic calibrators fit on Jul 2025 made August predictions *worse* on Portfolio C and D because the Jul bias pattern was in the opposite direction from Aug's. Aug 2024 is the calendar analog of the forecast target — same month, same position in the year — so any bias structure I fit there is more likely to transfer to Aug 2025.

The setup is leave-month-out: training excludes Aug 2024, validates on Aug 2024, then the final prediction model retrains on everything (including Aug 2024) and applies the calibrators to Aug 2025. Lag features naturally respect row-level temporal ordering even though the training window spans Aug 2024, so there's no real leakage.

### What stayed from earlier

- **Pooled XGBoost architecture.** Per-portfolio regressed every time I tried it.
- **`count:poisson` on Abandoned Calls** as the Stage-1 ABD target. Derived rate from `abd_calls / max(cv, 1)`.
- **Stage-2 abandon rate profile rescaled per day** to match Stage-1's derived daily rate.
- **Seasonal-naive blend for Abandoned Calls** with NB5 §4 weights (A=0.75, B=0.65, C=0.95, D=0.85). These weights were tuned on a Jul 2025 grid search against a quick XGBoost, so they're approximate for this configuration, but directionally right.

### What I removed

- **Monotone constraints.** The previous run showed they mildly regressed CV and ABD (0.2–0.5 pp worse WMAPE on 3 of 4 portfolios). The tree was already learning the right relationships from the data; the constraints were over-restricting edge-case splits without adding anything.

### Open caveats

- Aug 2024 might itself have idiosyncrasies that don't repeat in Aug 2025. Leave-month-out is the best I can do with the data I have, but it's not a guarantee.
- Lag-365 feature is NaN for Aug 2024 rows (no Aug 2023 data). XGBoost handles NaN natively so this isn't a crash, but it does mean the model can't use year-over-year signal for the Aug 2024 validation specifically.
- Calibrators fit on a model trained with Aug 2024 held out, applied to a model trained on everything. The two share most of their training data so the predictions should be close, but it's an approximation.
- Seasonal-naive blend weights are from NB5's Jul 2025 search. If I had more time I'd re-tune them on Aug 2024 holdout, but the seasonal-naive predictor I'd need for that (Aug 2023 × YoY ratio) doesn't exist because the data starts Jan 2024.

### Success criteria for this run

1. CV / CCT Aug 2024 holdout WMAPE in the 4–7% / 3–4% ballpark (comparable to prior Jul numbers)
2. Calibrators move Aug 2024 per-portfolio bias to near zero (by construction)
3. August 2025 derived abandon rate within ±25% of historical per portfolio
4. Submission: 0 NaN / 0 negatives / 0 mismatches

Last year's numbers aren't the only thing I care about — the real goal is that the calibration transfers to Aug 2025 predictions, which is hard to measure until someone releases Aug 2025 ground truth. Best I can do is sanity-check against the historical averages in cell 20.
